# AHP-LLM: Algorithm-Aware Tensor Adaptation Lab

**Research objective:** keep one model artifact fixed, extract representations from it, and adapt those representations for multiple downstream algorithms.

This notebook demonstrates the workflow with a small Transformer-shaped PyTorch model. It is a controlled stand-in for a trained LLM—not a pretrained or meaningfully trained language model. The same workflow can later load a real, frozen Hugging Face model artifact.

**Core experimental rule:** do not retrain or modify the base model between downstream use cases. Run inference, capture its hidden representations, and pass those representations through algorithm-specific adapters. Some downstream algorithms (for example, logistic regression or K-means) fit their own small downstream model; that does not mean the base LLM was retrained.

Important distinction:
- Model weights/configuration: fixed during the downstream comparison.
- Inference activations: dynamic; they depend on the input.
- Adapter: selected for the receiving algorithm's input contract.
- Permutation: changes axis order, not tensor values or rank.
- Pooling/projection: required when an algorithm needs a different rank or feature representation; permutation alone cannot do that.


## 0. Install dependencies

Core: `torch numpy scikit-learn matplotlib`. TensorFlow is optional for the Keras example.

```bash
python -m pip install torch numpy scikit-learn matplotlib
# Optional Keras section:
python -m pip install tensorflow
```

In [ ]:
import json, math, time, platform
from pathlib import Path
from itertools import permutations
import numpy as np
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", platform.python_version(), "| PyTorch:", torch.__version__, "| Device:", DEVICE)

## Experimental design: one fixed artifact, several algorithm adapters

The experiment uses one frozen base model for every use case:

1. Load the artifact once.
2. Freeze its parameters and run inference on a fixed evaluation corpus.
3. Capture the same hidden-state tensor for every downstream experiment.
4. Select an adapter based on the receiving algorithm's declared contract.
5. Compare downstream quality, latency, memory, and any required materialization.

The adapters are not all permutations. A convolution may need an axis permutation; logistic regression and clustering generally need a 2-D feature matrix, which requires sequence pooling or another feature-reduction step. A token-wise linear layer may consume `[B,S,H]` directly.

For a fair comparison, keep the base artifact, tokenizer, input examples, and extracted representations fixed. Record adapter configuration and random seeds. Do not interpret a faster tensor rearrangement as proof of faster end-to-end inference until the full downstream path is benchmarked.


## 1. Create a small Transformer-like artifact

This model is intentionally small so it can run on CPU. It produces a real hidden-state tensor with shape `[batch, sequence, hidden]`.

In [ ]:
class TinyTransformerArtifact(nn.Module):
    def __init__(self, vocab_size=1000, seq_len=32, d_model=48, nhead=4, layers=2):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model,
            batch_first=True, dropout=0.0, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, token_ids, return_hidden=False):
        x = self.token_embedding(token_ids) + self.position_embedding[:, :token_ids.shape[1], :]
        hidden = self.norm(self.encoder(x))
        return hidden if return_hidden else hidden.mean(dim=1)

model = TinyTransformerArtifact().to(DEVICE).eval()
tokens = torch.randint(0, 1000, (8, 32), device=DEVICE)
with torch.no_grad():
    hidden = model(tokens, return_hidden=True)
print("Hidden shape [B,S,H]:", tuple(hidden.shape))
print("Parameter count:", sum(p.numel() for p in model.parameters()))

## 2. Inspect artifact tensors and activation layout

`shape` describes logical dimensions. `stride` describes how those dimensions map to memory. `permute()` generally returns a view with reordered strides; it does not necessarily copy the tensor data.

In [ ]:
def inspect_tensor(name, t):
    return {
        "name": name, "shape": tuple(t.shape), "stride": tuple(t.stride()),
        "dtype": str(t.dtype), "device": str(t.device),
        "contiguous": t.is_contiguous(), "numel": t.numel()
    }

for name, tensor in list(model.state_dict().items())[:10]:
    print(name, tuple(tensor.shape), tensor.dtype)
print(inspect_tensor("hidden", hidden))

## 3. PyTorch `permute()` and axis permutations

For `[B,S,H]`, `permute(0,2,1)` produces `[B,H,S]`. The batch axis remains first. A rank-3 tensor has `3! = 6` possible axis orders; rank `n` has `n!`.

In [ ]:
x = hidden
y = x.permute(0, 2, 1)
print("Original:", inspect_tensor("x", x))
print("Permuted:", inspect_tensor("y", y))
assert tuple(x.shape) == (8, 32, 48)
assert tuple(y.shape) == (8, 48, 32)
assert torch.equal(y, x.transpose(1, 2))

def enumerate_permutations(shape):
    return [
        {"dims": p, "output_shape": tuple(shape[i] for i in p)}
        for p in permutations(range(len(shape)))
    ]

for item in enumerate_permutations((8, 32, 48)):
    print(item)
print("Rank-8 permutation count:", math.factorial(8))

## 4. A shape-contract planner

The downstream operation should declare the shape it expects. The planner finds a valid axis order, while preserving the batch axis if requested.

A permutation cannot change rank or dimension sizes. It only reorders existing axes.

In [ ]:
def find_permutations(input_shape, target_shape):
    if len(input_shape) != len(target_shape):
        return []
    return [
        p for p in permutations(range(len(input_shape)))
        if tuple(input_shape[i] for i in p) == tuple(target_shape)
    ]

def plan_permutation(input_shape, target_shape, keep_batch_first=True):
    input_shape, target_shape = tuple(input_shape), tuple(target_shape)
    if sorted(input_shape) != sorted(target_shape):
        raise ValueError(f"Permutation cannot change sizes: {input_shape} -> {target_shape}")
    candidates = find_permutations(input_shape, target_shape)
    if keep_batch_first:
        candidates = [p for p in candidates if p[0] == 0]
    if not candidates:
        raise ValueError("No valid permutation satisfies this shape contract.")
    return candidates[0]

dims = plan_permutation(hidden.shape, (8, 48, 32))
adapted = hidden.permute(*dims)
print("Chosen dims:", dims, "| result:", tuple(adapted.shape))

## 5. Adapt a Transformer representation for `Conv1d`

PyTorch `Conv1d` expects `[batch, channels, length]`. Our Transformer hidden states are `[batch, sequence, hidden]`. Permuting axes 1 and 2 adapts the layout.

In [ ]:
conv = nn.Conv1d(in_channels=48, out_channels=64, kernel_size=3, padding=1).to(DEVICE)
conv_input = hidden.permute(0, 2, 1)  # [B,H,S]
conv_output = conv(conv_input)
print("Transformer:", tuple(hidden.shape))
print("Conv1d input:", tuple(conv_input.shape))
print("Conv1d output:", tuple(conv_output.shape))
assert tuple(conv_output.shape) == (8, 64, 32)

## 6. Adapt Transformer representations for logistic regression

Logistic regression expects a 2D matrix `[samples, features]`, not a 3D tensor. Here we use mean pooling over the sequence axis to create `[B,H]`. Pooling is a separate operation; `permute()` alone cannot remove an axis.

In [ ]:
features_3d = hidden.detach().cpu().numpy()  # [B,S,H]
features = features_3d.mean(axis=1)          # [B,H] via sequence pooling
signal = features[:, :5].mean(axis=1)
labels = (signal > np.median(signal)).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.25, random_state=SEED, stratify=labels
)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, random_state=SEED))
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
print("Feature matrix:", features.shape)
print("Toy logistic-regression accuracy:", accuracy_score(y_test, pred))

## 7. Keras `Permute(dims=...)` semantics

Keras excludes the batch axis from `dims`. Thus `Permute((2,1))` converts `(batch, sequence, hidden)` into `(batch, hidden, sequence)`. PyTorch expresses the same operation as `x.permute(0,2,1)`.

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    inp = keras.Input(shape=(32, 48))
    out = layers.Permute((2, 1))(inp)
    keras_model = keras.Model(inp, out)
    print("Keras input:", keras_model.input_shape)
    print("Keras output:", keras_model.output_shape)
    sample = np.random.randn(4, 32, 48).astype("float32")
    print("Runtime output:", tuple(keras_model(sample).shape))
except ImportError:
    print("TensorFlow not installed; optional Keras demonstration skipped.")

## 8. Correctness tests

We verify that a permutation preserves element count and values, and that applying the inverse permutation restores the original tensor.

In [ ]:
def inverse_permutation(dims):
    inverse = [0] * len(dims)
    for new_axis, old_axis in enumerate(dims):
        inverse[old_axis] = new_axis
    return tuple(inverse)

def test_round_trip():
    sample = torch.arange(2*3*5).reshape(2,3,5)
    dims = (0,2,1)
    restored = sample.permute(*dims).permute(*inverse_permutation(dims))
    assert torch.equal(sample, restored)

def test_shape():
    sample = torch.randn(4,7,11)
    assert tuple(sample.permute(0,2,1).shape) == (4,11,7)

def test_incompatible_contract():
    try:
        plan_permutation((4,7,11), (4,77,1))
    except ValueError:
        return
    raise AssertionError("Expected invalid permutation contract to fail")

test_round_trip()
test_shape()
test_incompatible_contract()
print("All permutation tests passed.")

## 9. Benchmark view creation versus contiguous materialization

`permute()` is often a low-cost view operation. `.contiguous()` may allocate and copy data. A downstream kernel may perform differently depending on strides, so benchmark the complete operation rather than assuming a permutation makes a model faster.

In [ ]:
bench = torch.randn(64, 256, 256, device=DEVICE)

def average_ms(fn, repeats=50):
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(repeats):
        result = fn()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - start) * 1000 / repeats

view_ms = average_ms(lambda: bench.permute(0,2,1))
copy_ms = average_ms(lambda: bench.permute(0,2,1).contiguous())
print(f"permute view: {view_ms:.4f} ms/call")
print(f"permute + contiguous: {copy_ms:.4f} ms/call")

## 10. Save the model artifact and permutation metadata

The saved metadata records the source and target layouts. If you permanently insert a permutation into a model's forward pass, save that updated model definition/configuration too—not only the weights.

In [ ]:
artifact_dir = Path("ahp_tensor_permutation_artifact")
artifact_dir.mkdir(exist_ok=True)
torch.save(model.state_dict(), artifact_dir / "tiny_transformer_state.pt")

metadata = {
    "model_type": "TinyTransformerArtifact",
    "framework": "pytorch",
    "source_layout": ["batch", "sequence", "hidden"],
    "target_layout": ["batch", "hidden", "sequence"],
    "pytorch_permutation": [0, 2, 1],
    "keras_permute_dims": [2, 1],
    "note": "Axis adaptation only; no model retraining performed."
}
(artifact_dir / "permutation_metadata.json").write_text(json.dumps(metadata, indent=2))
print("Saved:", [p.name for p in artifact_dir.iterdir()])

## 11. Freeze one artifact and reuse its representations

The current `TinyTransformerArtifact` is a small randomly initialized demonstration model. We freeze it here to simulate the fixed-artifact experiment. When replacing it with a real trained LLM, load the trained checkpoint and tokenizer in this same stage; do not train it separately for each downstream algorithm.

We use a fixed synthetic token batch so every downstream algorithm receives the same base representations. In a real experiment, replace this with a fixed evaluation corpus and the actual tokenizer.

In [ ]:
# Freeze the single base artifact for all downstream use cases.
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)

BASE_ARTIFACT_ID = "tiny-transformer-demo-v1"
BASE_ARTIFACT_FROZEN = all(not p.requires_grad for p in model.parameters())
assert BASE_ARTIFACT_FROZEN

# Reuse the same input batch and the same extracted hidden states for every adapter.
torch.manual_seed(17)
fixed_token_batch = torch.randint(0, 1000, (8, 32), device=DEVICE)

with torch.inference_mode():
    fixed_hidden = model(fixed_token_batch, return_hidden=True)  # [B, S, H]

print("Artifact:", BASE_ARTIFACT_ID)
print("Frozen:", BASE_ARTIFACT_FROZEN)
print("Hidden representation:", tuple(fixed_hidden.shape))


## 12. Algorithm adapter catalog: same representations, different consumers

This catalog demonstrates four downstream use cases:

- **Conv1D:** `[B,S,H]` → `[B,H,S]` using `permute`.
- **Logistic regression:** sequence pooling → `[B,H]`, then fit a downstream classifier.
- **K-means:** sequence pooling → `[B,H]`, then cluster representations.
- **Token-wise linear projection:** consume `[B,S,H]` directly; no permutation required.

The base model remains frozen. Logistic regression and K-means fit downstream components on the fixed representations. This is distinct from retraining the LLM.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Convert the exact same frozen-model activations to CPU for sklearn use.
# Mean pooling removes the sequence axis; it is not a permutation.
representations = fixed_hidden.detach().float().cpu()
pooled_features = representations.mean(dim=1).numpy()  # [B, H]

# A. Conv1D adapter: preserve batch, move hidden/channels before sequence/length.
conv_input = fixed_hidden.permute(0, 2, 1)  # [B, H, S]
conv1d_adapter = nn.Conv1d(
    in_channels=conv_input.shape[1], out_channels=24, kernel_size=3, padding=1
).to(DEVICE)
with torch.inference_mode():
    conv_output = conv1d_adapter(conv_input)
assert conv_output.shape[0] == fixed_hidden.shape[0]
print("Conv1D input/output:", tuple(conv_input.shape), tuple(conv_output.shape))

# B. Logistic regression adapter: pooled [B,H] vectors.
# Synthetic labels only demonstrate the interface; real work needs task labels.
labels = np.array([0, 1, 0, 1, 0, 1, 0, 1])
X_train, X_test, y_train, y_test = train_test_split(
    pooled_features, labels, test_size=0.25, random_state=7, stratify=labels
)
logreg_adapter = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))
logreg_adapter.fit(X_train, y_train)
logreg_predictions = logreg_adapter.predict(X_test)
print("Logistic regression feature shape:", pooled_features.shape)
print("Demo holdout accuracy:", accuracy_score(y_test, logreg_predictions))

# C. K-means adapter: same pooled features, unsupervised clustering.
kmeans_adapter = KMeans(n_clusters=2, random_state=7, n_init=10)
cluster_ids = kmeans_adapter.fit_predict(pooled_features)
print("K-means input shape:", pooled_features.shape)
print("Cluster assignments:", cluster_ids.tolist())

# D. Token-wise linear projection: [B,S,H] is already a valid input layout.
token_projection = nn.Linear(fixed_hidden.shape[-1], 16).to(DEVICE)
with torch.inference_mode():
    token_features = token_projection(fixed_hidden)
print("Token projection:", tuple(token_features.shape))


## 13. Record the experiment contract

The record ties every downstream result to the same base artifact. In a larger experiment, persist this metadata beside metrics and checkpoints so comparisons are reproducible.

In [ ]:
experiment_record = {
    "base_artifact_id": BASE_ARTIFACT_ID,
    "base_artifact_frozen": BASE_ARTIFACT_FROZEN,
    "input_token_batch_shape": list(fixed_token_batch.shape),
    "hidden_state_layout": ["batch", "sequence", "hidden"],
    "hidden_state_shape": list(fixed_hidden.shape),
    "downstream_adapters": {
        "conv1d": {
            "operation": "permute",
            "pytorch_dims": [0, 2, 1],
            "output_layout": ["batch", "channels", "length"],
            "output_shape": list(conv_input.shape),
        },
        "logistic_regression": {
            "operation": "mean_pool_sequence",
            "output_layout": ["samples", "features"],
            "output_shape": list(pooled_features.shape),
            "base_llm_retrained": False,
        },
        "kmeans": {
            "operation": "mean_pool_sequence",
            "output_layout": ["samples", "features"],
            "output_shape": list(pooled_features.shape),
            "base_llm_retrained": False,
        },
        "token_projection": {
            "operation": "direct_layout",
            "output_layout": ["batch", "sequence", "features"],
            "output_shape": list(token_features.shape),
        },
    },
}
print(json.dumps(experiment_record, indent=2))
(Path("ahp_tensor_permutation_artifact") / "multi_algorithm_experiment.json").write_text(
    json.dumps(experiment_record, indent=2)
)


## 14. Expandable algorithm catalog with a common adapter interface

This section adds a small, extensible catalog. Each entry declares:
- task family (classification, regression, clustering, dimensionality reduction, anomaly detection, or tensor operation)
- expected input rank/layout
- adapter strategy
- estimator factory (where applicable)

The common adapter converts the frozen model's `[B,S,H]` representation into the input form expected by the selected algorithm. For scikit-learn estimators, mean pooling produces `[samples, features]`. The Conv1D path remains a PyTorch tensor operation and uses axis permutation.

The examples use a separate fixed batch of 64 synthetic token sequences to make the demonstration less fragile than fitting models on only eight samples. Every algorithm receives representations from the same frozen base artifact and the same fixed input batch.

In [ ]:
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, IsolationForest
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, mean_squared_error, silhouette_score

# Snapshot weights so we can verify that downstream experiments do not alter the base artifact.
base_weights_before = {
    name: value.detach().cpu().clone()
    for name, value in model.state_dict().items()
}

# A fixed input corpus: all downstream algorithms consume representations from this same batch.
torch.manual_seed(2026)
catalog_token_batch = torch.randint(0, 1000, (64, 32), device=DEVICE)
with torch.inference_mode():
    catalog_hidden = model(catalog_token_batch, return_hidden=True)  # [B,S,H]

def adapt_representation(hidden_states: torch.Tensor, strategy: str):
    """Convert [B,S,H] hidden states into a declared downstream input layout."""
    if strategy == "mean_pool":
        return hidden_states.mean(dim=1).detach().float().cpu().numpy()  # [B,H]
    if strategy == "max_pool":
        return hidden_states.max(dim=1).values.detach().float().cpu().numpy()  # [B,H]
    if strategy == "bhs_permute":
        return hidden_states.permute(0, 2, 1)  # [B,H,S], torch tensor view
    if strategy == "identity":
        return hidden_states  # [B,S,H]
    raise ValueError(f"Unknown adapter strategy: {strategy}")

ALGORITHM_CATALOG = {
    "logistic_regression": {
        "task": "classification",
        "input_rank": 2,
        "input_layout": ["samples", "features"],
        "adapter": "mean_pool",
        "factory": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)),
        "requires_labels": True,
    },
    "random_forest_classifier": {
        "task": "classification",
        "input_rank": 2,
        "input_layout": ["samples", "features"],
        "adapter": "mean_pool",
        "factory": lambda: RandomForestClassifier(n_estimators=100, random_state=42),
        "requires_labels": True,
    },
    "svc": {
        "task": "classification",
        "input_rank": 2,
        "input_layout": ["samples", "features"],
        "adapter": "mean_pool",
        "factory": lambda: make_pipeline(StandardScaler(), SVC()),
        "requires_labels": True,
    },
    "knn": {
        "task": "classification",
        "input_rank": 2,
        "input_layout": ["samples", "features"],
        "adapter": "mean_pool",
        "factory": lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
        "requires_labels": True,
    },
    "ridge_regression": {
        "task": "regression",
        "input_rank": 2,
        "input_layout": ["samples", "features"],
        "adapter": "mean_pool",
        "factory": lambda: make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
        "requires_labels": True,
    },
    "random_forest_regressor": {
        "task": "regression",
        "input_rank": 2,
        "input_layout": ["samples", "features"],
        "adapter": "mean_pool",
        "factory": lambda: RandomForestRegressor(n_estimators=100, random_state=42),
        "requires_labels": True,
    },
    "kmeans": {
        "task": "clustering",
        "input_rank": 2,
        "input_layout": ["samples", "features"],
        "adapter": "mean_pool",
        "factory": lambda: KMeans(n_clusters=3, n_init=10, random_state=42),
        "requires_labels": False,
    },
    "pca": {
        "task": "dimensionality_reduction",
        "input_rank": 2,
        "input_layout": ["samples", "features"],
        "adapter": "mean_pool",
        "factory": lambda: PCA(n_components=8, random_state=42),
        "requires_labels": False,
    },
    "isolation_forest": {
        "task": "anomaly_detection",
        "input_rank": 2,
        "input_layout": ["samples", "features"],
        "adapter": "mean_pool",
        "factory": lambda: IsolationForest(n_estimators=100, contamination=0.1, random_state=42),
        "requires_labels": False,
    },
    "conv1d": {
        "task": "tensor_operation",
        "input_rank": 3,
        "input_layout": ["batch", "channels", "length"],
        "adapter": "bhs_permute",
        "factory": None,  # Created from the runtime channel count.
        "requires_labels": False,
    },
}

def get_algorithm_input(hidden_states: torch.Tensor, algorithm_name: str):
    """Apply the adapter declared by the selected catalog entry and validate rank."""
    spec = ALGORITHM_CATALOG[algorithm_name]
    adapted = adapt_representation(hidden_states, spec["adapter"])
    actual_rank = adapted.ndim if isinstance(adapted, torch.Tensor) else np.asarray(adapted).ndim
    if actual_rank != spec["input_rank"]:
        raise ValueError(
            f"{algorithm_name} expects rank {spec['input_rank']}, got rank {actual_rank}"
        )
    return adapted

print(f"Catalog entries: {len(ALGORITHM_CATALOG)}")
for name, spec in ALGORITHM_CATALOG.items():
    print(f"{name:28s} task={spec['task']:24s} adapter={spec['adapter']} layout={spec['input_layout']}")


## 15. Run the catalog against the same frozen representations

For supervised examples, the labels below are synthetic demonstration targets. In a real task, replace them with ground-truth labels or values. Unsupervised algorithms do not consume labels.

The scores are plumbing checks, not evidence that the randomly initialized demonstration Transformer has useful learned representations.

In [ ]:
# Every catalog entry starts from this exact same pooled feature matrix.
X_catalog = get_algorithm_input(catalog_hidden, "logistic_regression")
assert X_catalog.shape == (64, catalog_hidden.shape[-1])

# Deterministic synthetic labels and regression targets for demonstrating estimator interfaces.
token_means = catalog_token_batch.float().mean(dim=1).detach().cpu().numpy()
y_class = (token_means > np.median(token_means)).astype(int)
feature_weights = np.linspace(-0.5, 0.5, X_catalog.shape[1], dtype=np.float32)
y_regression = X_catalog @ feature_weights + 0.05 * np.random.default_rng(42).normal(size=64)

X_train, X_test, y_train, y_test = train_test_split(
    X_catalog, y_class, test_size=0.25, random_state=42, stratify=y_class
)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_catalog, y_regression, test_size=0.25, random_state=42
)

catalog_results = {}

# Classification algorithms share the same frozen-model features and same train/test split.
for algorithm_name in (
    "logistic_regression", "random_forest_classifier", "svc", "knn"
):
    spec = ALGORITHM_CATALOG[algorithm_name]
    estimator = spec["factory"]()
    estimator.fit(X_train, y_train)
    predictions = estimator.predict(X_test)
    catalog_results[algorithm_name] = {
        "task": spec["task"],
        "accuracy": float(accuracy_score(y_test, predictions)),
        "input_shape": list(X_catalog.shape),
        "adapter": spec["adapter"],
    }

# Regression algorithms share the same features and target.
for algorithm_name in ("ridge_regression", "random_forest_regressor"):
    spec = ALGORITHM_CATALOG[algorithm_name]
    estimator = spec["factory"]()
    estimator.fit(Xr_train, yr_train)
    predictions = estimator.predict(Xr_test)
    catalog_results[algorithm_name] = {
        "task": spec["task"],
        "rmse": float(mean_squared_error(yr_test, predictions) ** 0.5),
        "input_shape": list(X_catalog.shape),
        "adapter": spec["adapter"],
    }

# Unsupervised consumers use the same X_catalog matrix, without labels.
kmeans = ALGORITHM_CATALOG["kmeans"]["factory"]()
cluster_ids = kmeans.fit_predict(X_catalog)
catalog_results["kmeans"] = {
    "task": "clustering",
    "cluster_count": int(len(np.unique(cluster_ids))),
    "input_shape": list(X_catalog.shape),
    "adapter": "mean_pool",
}
pca = ALGORITHM_CATALOG["pca"]["factory"]()
pca_features = pca.fit_transform(X_catalog)
catalog_results["pca"] = {
    "task": "dimensionality_reduction",
    "output_shape": list(pca_features.shape),
    "explained_variance_ratio_sum": float(pca.explained_variance_ratio_.sum()),
    "adapter": "mean_pool",
}
isolation_forest = ALGORITHM_CATALOG["isolation_forest"]["factory"]()
anomaly_labels = isolation_forest.fit_predict(X_catalog)
catalog_results["isolation_forest"] = {
    "task": "anomaly_detection",
    "outlier_count": int((anomaly_labels == -1).sum()),
    "input_shape": list(X_catalog.shape),
    "adapter": "mean_pool",
}

# Tensor operation uses a rank-3 adapter instead of a scikit-learn feature matrix.
conv_input_catalog = get_algorithm_input(catalog_hidden, "conv1d")
conv_layer_catalog = nn.Conv1d(
    in_channels=conv_input_catalog.shape[1], out_channels=24, kernel_size=3, padding=1
).to(DEVICE)
with torch.inference_mode():
    conv_output_catalog = conv_layer_catalog(conv_input_catalog)
catalog_results["conv1d"] = {
    "task": "tensor_operation",
    "input_shape": list(conv_input_catalog.shape),
    "output_shape": list(conv_output_catalog.shape),
    "adapter": "bhs_permute",
}

# Confirm downstream fitting did not modify the frozen base model.
assert all(
    torch.equal(base_weights_before[name], value.detach().cpu())
    for name, value in model.state_dict().items()
)
print(json.dumps(catalog_results, indent=2))
print("\nBase artifact weights unchanged:", True)


## 16. Persist the algorithm catalog and run results

The catalog metadata is saved separately from the model weights. This makes the downstream experiment configuration auditable and allows new algorithms to be added without changing the base artifact.

In [ ]:
catalog_metadata = {
    "base_artifact_id": BASE_ARTIFACT_ID,
    "base_artifact_frozen": BASE_ARTIFACT_FROZEN,
    "base_hidden_layout": ["batch", "sequence", "hidden"],
    "fixed_input_batch_shape": list(catalog_token_batch.shape),
    "shared_feature_shape": list(X_catalog.shape),
    "algorithms": {
        name: {
            key: value
            for key, value in spec.items()
            if key != "factory"
        }
        for name, spec in ALGORITHM_CATALOG.items()
    },
    "results": catalog_results,
}
catalog_path = Path("ahp_tensor_permutation_artifact") / "algorithm_catalog_results.json"
catalog_path.write_text(json.dumps(catalog_metadata, indent=2))
print("Saved:", catalog_path)


## 17. Interpretation and next research step

This experiment isolates the design you described: **one fixed model artifact, reused across multiple algorithm-specific pathways**.

What it establishes:
- The same frozen model can provide representations to several downstream algorithms.
- Each receiving algorithm can declare a tensor contract.
- The adapter can choose a valid permutation when dimensions already match, or a separate operation such as pooling when they do not.
- Downstream models may be fitted independently without changing the base LLM.

What it does not establish:
- That permutation improves model quality.
- That the adaptation engine reduces LLM training time.
- That one layout is fastest on every device.
- That this randomly initialized demo Transformer is a trained LLM.

Next experiment: replace the demo model with one genuinely pretrained, frozen open-weight LLM; extract hidden states for a fixed dataset; run all adapters against those identical representations; then benchmark quality, p50/p95 latency, peak memory, and materialization cost. Only after that should the engine select layouts automatically based on measured results.


## 18. Original permutation findings

### This notebook demonstrates
- How to inspect a model artifact and its activations.
- How to enumerate and validate dimension permutations.
- How Keras `Permute` and PyTorch `permute()` differ in batch-axis notation.
- How to adapt `[B,S,H]` to `[B,H,S]` for `Conv1d`.
- Why logistic regression needs pooling/projection to become a 2D feature matrix.
- Why permutation and physical memory copying have different costs.

### It does not demonstrate
- Instant LLM training.
- New learned knowledge from changing axis order.
- That an arbitrary target curve is a valid tensor-layout objective.
- That permutation alone improves quality or training speed.

### Next experiment
Build a layout selector that benchmarks several valid layouts for a specific downstream workload and selects the fastest one on the target hardware. That would test a practical form of algorithm-aware tensor permutation.

## 19. Exercises

1. Change `d_model` to 64 and inspect the resulting hidden-state shape.
2. Write a helper that maps `BSH` to `BHS`.
3. Compare `transpose(1,2)` with `permute(0,2,1)`.
4. Feed `[B,S,H]` directly to `Conv1d`, inspect the error, and correct it.
5. Compare end-to-end convolution latency for contiguous and non-contiguous inputs.
6. Compare mean pooling and max pooling before logistic regression.

**Research question:** Can a controller reliably select the fastest valid tensor layout for a particular operation and hardware target?